In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import os
import sys
import tarfile

import boto3
import pandas as pd

MODEL_ID = "usecase2"
TRAINING_MODE = "2"  # 1: preliminary; 2: full using the saved metadata below.

# Every input comes from S3; no existing local data directory is used.
S3_INPUTS = {
    "prepared_data": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/prepared/data.csv",
    "selection": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/selection/selected_features.csv",
    "parameters": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/reports/evaluation/parameters.json",
    "metadata": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/reports/evaluation/model_metadata.json"
}

# This checkout supplies the Python code you are editing, not the input data.
REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))

# Fresh local copies are needed because the step accepts filesystem paths.
work_directory = Path(mkdtemp(prefix="train-"))
data_directory = work_directory / "data"
output_directory = work_directory / "output"
model_directory = work_directory / "model"
selection_file = work_directory / "selection/selected_features.csv"
parameter_path = work_directory / "settings" / MODEL_ID / "parameters.json"
metadata_path = parameter_path.with_name("model_metadata.json")

output_directory.mkdir()
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "MLOPS_TRAINING_MODE": TRAINING_MODE,
})

# Use downloaded metadata beside parameters.json, without an S3 URI override.
os.environ.pop("MLOPS_MODEL_METADATA_URI", None)

# Download the exact files listed above without transforming them.
s3 = boto3.client("s3", region_name="us-east-1")
local_files = {
    'prepared_data': data_directory / "data.csv",
    'selection': selection_file,
    'parameters': parameter_path,
    'metadata': metadata_path,
}
for name, local_file in local_files.items():
    local_file.parent.mkdir(parents=True, exist_ok=True)
    address = urlsplit(S3_INPUTS[name])
    s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))

parameters = json.loads(parameter_path.read_text())
metadata = json.loads(metadata_path.read_text())


In [ ]:
# Optional inspection; the following cell executes the step.
print("Downloaded from:", S3_INPUTS)
print("New local working directory:", work_directory)
print("Local experiment output:", output_directory)
display(pd.read_csv(data_directory / "data.csv", nrows=10))


In [ ]:
from mlops_models.model_base.training import train

train = importlib.reload(train)
result_directory = train.train_model(
    data_directory, selection_file, model_directory, output_directory, parameter_path,
)
result_directory
